# Mini Trabalho 3: Exploração dos Dados — Saneamento e Perfil Demográfico

Nesta fase, o objetivo é auditar a base de dados escolhida para o projeto. Antes de pensar em qualquer algoritmo, precisamos entender quem são esses pacientes e se o dado é íntegro. Será investigado investigado anomalias em identificadores, validar se a distribuição de idade faz sentido clínico e, principalmente, checar se o nível de escolaridade pode estar camuflando sinais precoces de Alzheimer. 

O foco dessa etapa é preparar o terreno, visto que sem uma base limpa, o modelo pode ter muito underfitting, que é quando o modelo é tão sensível que só chuta valores sem uma base sólida de predição.

A seguir, estão as bibliotecas necessárias para gerar os gráficos e manipular o dataframe:

## Bibliotecas Usadas e Configuração Inicial

A seguir, estão as bibliotecas necessárias para gerar os gráficos e manipular o dataframe:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")

### Carga e Tradução Semântica

O arquivo bruto usa inteiros para quase tudo. É bastante útil e eficiente para a máquina, mas péssimo para a análise humana. Porém, na etapa 2, foi gerado um dicionário de dados que ajudam os integrantes do grupo a entender essas variáveis, e esse dicionário será usado aqui.

In [ ]:
def leitura_df():
    
    path_arquivo = 'data/alzheimers_disease_data.csv'
    if not os.path.exists(path_arquivo):
        path_arquivo = '../data/alzheimers_disease_data.csv'

    df_bruto = pd.read_csv(path_arquivo)

    df_bruto['Genero_Label'] = df_bruto['Gender'].map({0: 'Masculino', 1: 'Feminino'})
    df_bruto['Etnia_Label'] = df_bruto['Ethnicity'].map({
        0: 'Caucasiano', 1: 'Afrodescendente', 2: 'Asiático', 3: 'Outro'
    })
    df_bruto['Escolaridade_Label'] = df_bruto['EducationLevel'].map({
        0: 'Nenhum', 1: 'Ensino Médio', 2: 'Graduação', 3: 'Pós-Graduação'
    })
    df_bruto['Status_Diagnostico'] = df_bruto['Diagnosis'].map({0: 'Saudável', 1: 'Alzheimer'})

    return df_bruto

df = leitura_df()

### Verificação de Integridade

Nessa etapa será verificado se há valores nulos, repetidos ou variáveis descartáveis. Mesmo que o dataset usado do kaggle seja sintético, essa etapa é fundamental em uma base de dados real, já que é imprudente assumir que os dados estarão limpos, especialmente dados do domínio da saúde.

In [ ]:
def verificacao(dataframe):
    print(f"Total de registros na base: {len(dataframe)}")
    print(f"Registros duplicados (IDs): {dataframe['PatientID'].duplicated().sum()}")
    print(f"Valores nulos: {dataframe.isnull().sum().sum()}")
    print(f"Diversidade de Médicos: {dataframe['DoctorInCharge'].nunique()}") 

verificacao(df)

### Resultado da Verificação

Foi possível concluir que não há valores nulos ou repetidos no dataset. Outro ponto importante, é que a coluna do médico responsável tem apenas um valor placeholder chamado XXXConfid. Com base nisso, é fácil concluir que essa feature é ruído e não serve para a modelagem, pois pode pesar no treinamento, e dependendo pode causar overfitting. Um exemplo tátil que eu uso para explicar o overfitting por ruído de feature é ter uma feature de cpf no dataset. A inteligência Artificial poderia apontar que quem termina o cpf com 210 teria mais chance de ter alzheimer que o resto, sendo que é fácil concluir por análise humana que tal afirmação é incondizente com a realidade.

### Gênero e Etnia

É necessário saber quem são os pacientes. Um desequilíbrio gritante aqui pode enviesar todo o sistema de IA (nesse caso, precisaria utilizar métodos de sampling, como SMOTE, oversampling, etc). A seguir, veremos como o Alzheimer se distribui nesses grupos demográficos. O ponto aqui é que o modelo precisa representar todos.

In [ ]:
plt.figure(figsize=(15, 6))
# genero
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='Genero_Label', hue='Status_Diagnostico', palette='muted')
plt.title('Distribuição por Gênero')
plt.xlabel('Sexo')
plt.ylabel('Pacientes')

# etnia
plt.subplot(1, 2, 2)
sns.countplot(data=df, x='Etnia_Label', hue='Status_Diagnostico', palette='deep')
plt.title('Distribuição por Etnia')
plt.xlabel('Etnia')
plt.ylabel('')

plt.tight_layout()
plt.show()

### Escolaridade e Cognição

Aqui mora uma nuance clínica importante, pois pacientes mais instruídos podem compensar perdas de memória em testes básicos de consultório, mesmo que eles estejam em fase inicial do Alzheimer. O gráfico a seguir vai ajudar a visualizar se o score mental do MMSE cai de forma consistente ou se o nível de estudo mascara a gravidade real da patologia.

In [ ]:
plt.figure(figsize=(12, 7))
sns.boxplot(data=df, x='Escolaridade_Label', y='MMSE', hue='Status_Diagnostico', palette='coolwarm')
plt.title('Score MMSE vs Nível de Estudo')
plt.ylabel('Pontuação MMSE')
plt.xlabel('Grau de Instrução')
plt.axhline(15, linestyle='--', color='red', alpha=0.5)
plt.axhline(20, linestyle='--', color='green', alpha=0.5)
plt.show()

### Idade e Equilíbrio das Classes

Como o Alzheimer é uma doença degenerativa, o tempo é o maior fator de risco. Será testado a hipótese de que a curva de idade da base reflete esse padrão biológico. Ademais, precisa de pessoas que tenham Alzheimer suficientes para o treinamento, porque Se a base estiver muito desequilibrada, pode cair no caso de víes, seja overfitting ou underfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.histplot(data=df, x='Age', hue='Status_Diagnostico', kde=False, ax=axes[0], palette='magma')
axes[0].set_title('Densidade de Idade')
axes[0].set_xlabel('Anos')
# vê desequilibrio
sns.countplot(data=df, x='Status_Diagnostico', palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Balanço: Saudáveis vs Alzheimer')
axes[1].set_xlabel('Diagnóstico Final')
plt.tight_layout()
plt.show()

### Observação sobre relação entre MMSE, Escolaridade, e Balanço entre Saudáveis e não Saudáveis

Vale salientar, que embora a literatura médica diga que um teste satisfatório do MMSE tenha que ser pontuação >= 25, a média geral desse dataset sintético é 14. 76, mesmo maior parte da população seja considerada saudável no dataset. Isso pode indicar subnotificação de demência, onde esses indivíduos podem estar com comprometimento cognitivo leve (CCL). Por tal motivo, foi decidido, arbitrariamente, classificar como satisfatório, todo teste cognitivo (MMSE) com pontuações >= 20, sem seguir uma base científica robusta (para fins de estudo). Ainda sim, seguindo a literatura médica, foi classificada toda pontuação menor que 16 como comprometimento cognitivo grave. A seguir, há um script simples em python que diz a média, o mínimo e o máximo da pontuação do MMSE, só para confirmar a análise.

In [ ]:
import pandas as pd
import os

caminho = 'data/alzheimers_disease_data.csv'
if not os.path.exists( caminho):
    caminho = '../data/alzheimers_disease_data.csv'
df = pd.read_csv(caminho, usecols = ['MMSE'])
media = df['MMSE'].mean()
minimo = df['MMSE'].min()
maximo = df['MMSE'].max()

print(f"Estatísticas do MMSE do dataset:")
print(f"Média:  {media:.2f}")
print(f"Mínimo: {minimo:.2f}")
print(f"Máximo: {maximo:.2f}")


### Conclusão e Próximos Passos

Para a próxima etapa de modelagem, as diretrizes são claras. Primeiro, o descarte de PatientID e DoctorInCharge é mandatório. IDs não ensinam nada ao modelo e o médico é uma constante que gera peso morto. 

Temos cerca de 35% de casos positivos. O desequilíbrio existe, mas não é desesperador. Isso significa que focar apenas em acurácia seria um erro. Será preciso utilizar métricas como Recall, precisão, e e F1-Score (embora F1-score seja a média harmônica entre os dois, ainda é fundamental analisar esses dados separadamente para melhor análise dos resultados). A performance funcional é o preditor principal, mas as nuances da escolaridade mostram que precisaremos cruzar esses dados com indicadores biométricos para não ignorar possíveis pessoas com Alzheimer que não seriam pegas em um teste cognitivo básico. 